# CP4 · Notebook 03 — Dataset (EDA + DataLoader)

Exploración del dataset: distribución de steering, visualización, augmentation, DataLoader PyTorch. ~8 min.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path

data = np.load('../datasets/cp4-highway-bc.npz')
for k in data.files:
    print(f'  {k:18s}  shape={data[k].shape}  dtype={data[k].dtype}')

## 1. Histograma de steering — el sesgo a 0

Una autopista nominal tiene **mucho "recto"** y poco giro. Si la CNN no contrarresta esto, aprenderá un sesgo de inacción.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (split, color) in zip(axes,
        [('train', '#4a6fa5'), ('val_in', '#f4a261'), ('val_ood', '#DA4544')]):
    a = data[f'{split}_actions']
    ax.hist(a, bins=30, color=color, edgecolor='black', alpha=0.8)
    ax.set_title(f'{split}  N={len(a)}  mean={a.mean():+.3f}  std={a.std():.3f}')
    ax.set_xlabel('steering [-1, 1]')
    ax.axvline(0, color='black', linestyle=':', alpha=0.4)
plt.tight_layout(); plt.show()

# Cuántos % son "casi 0"?
for split in ['train','val_in','val_ood']:
    a = data[f'{split}_actions']
    pct = 100 * (np.abs(a) < 0.05).mean()
    print(f'  {split:8s}: {pct:.1f}% con |steering| < 0.05')

**Observación**: si >50% del dataset tiene `|steering| < 0.05`, la CNN puede aprender a predecir **siempre 0** y obtener buen MSE. Es el bug clásico de BC.

**Mitigación habitual**: oversampling de muestras con steering grande, o **weighted MSE** penalizando errores en muestras de giro.

## 2. Visualizar 6 observaciones de cada split

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for row, split in enumerate(['train','val_in','val_ood']):
    obs_split = data[f'{split}_obs']
    act_split = data[f'{split}_actions']
    np.random.seed(0)
    idx = np.random.choice(len(obs_split), 6, replace=False)
    for col, i in enumerate(idx):
        # Tomar el frame más reciente (canal 2 = más actual del stack)
        axes[row, col].imshow(obs_split[i, :, :, 2], cmap='gray')
        axes[row, col].set_title(f'{split} idx={i}  s={act_split[i]:+.2f}', fontsize=8)
        axes[row, col].axis('off')
plt.tight_layout(); plt.show()

## 3. Data augmentation: flip horizontal

Conducir es **simétrico izquierda/derecha**. Podemos **doblar** efectivamente el dataset flipeando la observación horizontalmente y **negando** el steering. Truco clásico de PilotNet.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class HighwayBCDataset(Dataset):
    def __init__(self, obs, actions, augment=False):
        self.obs = obs              # (N, 84, 84, 3) uint8
        self.actions = actions      # (N,) float32
        self.augment = augment
    def __len__(self): return len(self.obs)
    def __getitem__(self, idx):
        x = self.obs[idx].astype(np.float32) / 255.0   # (84, 84, 3) → [0,1]
        y = self.actions[idx]
        if self.augment and np.random.rand() < 0.5:
            x = x[:, ::-1, :].copy()   # flip horizontal
            y = -y
        x = torch.from_numpy(x.transpose(2, 0, 1))     # (3, 84, 84)
        return x, torch.tensor(y, dtype=torch.float32)

train_ds = HighwayBCDataset(data['train_obs'], data['train_actions'], augment=True)
val_in_ds = HighwayBCDataset(data['val_in_obs'], data['val_in_actions'], augment=False)
val_ood_ds = HighwayBCDataset(data['val_ood_obs'], data['val_ood_actions'], augment=False)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
val_in_loader = DataLoader(val_in_ds, batch_size=64, shuffle=False, num_workers=0)
val_ood_loader = DataLoader(val_ood_ds, batch_size=64, shuffle=False, num_workers=0)

print(f'Train: {len(train_ds)} (con augmentation flip horizontal)')
print(f'Val in-dist: {len(val_in_ds)}')
print(f'Val OOD:     {len(val_ood_ds)}')

# Sanity check: una batch
x, y = next(iter(train_loader))
print(f'Batch shape: x={x.shape}  y={y.shape}')
print(f'  x range:   [{x.min():.2f}, {x.max():.2f}]')
print(f'  y range:   [{y.min():+.2f}, {y.max():+.2f}]')

## 4. Guardar los DataLoaders para 04

Persistimos referencias (no los objetos en sí — Python no los serializa) para que `04` use los mismos splits.

In [ ]:
import json
OUT = Path('../outputs'); OUT.mkdir(exist_ok=True)
stats = {
    'train_size': len(train_ds),
    'val_in_size': len(val_in_ds),
    'val_ood_size': len(val_ood_ds),
    'train_steering_mean': float(data['train_actions'].mean()),
    'train_steering_std': float(data['train_actions'].std()),
    'train_zero_pct': float(100 * (np.abs(data['train_actions']) < 0.05).mean()),
    'augmentation': 'horizontal_flip',
}
with open(OUT / '03_dataset_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)
print(f'✅ Stats guardadas en {OUT / "03_dataset_stats.json"}')
print('   Ve a 04_modelo_pilotnet.ipynb')